# EDA и бинарная классификация на датасете «Титаник»

**Дисциплина:** Машинное обучение и ИИ  
**Модуль 1:** Введение в Data Science и EDA  
**Датасет:** Titanic  

**Выполнил:** Емельянова Ксения Павловна  
**Группа:** ПКТб-23-1  
**Преподаватель:** Спирин Игорь Сергеевич  

**Ссылка на репозиторий:** https://github.com/fivifri/ml-course-emelyanova  
**Путь к модулю:** `module-1-titanic/notebook.ipynb`  
**Дата сдачи:** 2026-28-09

## Введение

**Задача:** бинарная классификация — предсказать, выжил ли пассажир Titanic (0 — не выжил, 1 — выжил).

**Что такое ML-проект:** ML-проект — это полный жизненный цикл создания интеллектуального решения, который начинается с анализа бизнеса и сбора данных, продолжается их подготовкой, EDA и созданием признаков, а завершается обучением, подбором моделей, их оценкой и внедрением. Это итеративный процесс, где результаты оценки часто заставляют возвращаться к начальным этапам для улучшения качества.

**Почему классификация, а не регрессия:** Целевая переменная является дискретной и принимает значения из ограниченного множества классов (выжил / не выжил), а не непрерывным числовым значением (как, например, цена квартиры или температура). Регрессия предсказывает непрерывную величину, тогда как классификация разделяет объекты по категориям.

**Какая метрика важнее:** В этой задаче наиболее информативной метрикой является **ROC-AUC**, так как она оценивает способность модели ранжировать пассажиров по вероятности выживания независимо от выбранного порога классификации и устойчива к возможному дисбалансу классов. Для оценки качества на фиксированном пороге лучше использовать **F1-score**, поскольку она гармонично объединяет точность (Precision) и полноту (Recall), в отличие от Accuracy, которая может давать обманчиво высокий результат при дисбалансе.

**Цель работы:**
1. Провести полный EDA датасета Титаник.
2. Обучить 2+ модели и достичь ROC-AUC ≥ 0.80.
3. Реализовать функцию предсказания для нового пассажира.

## Теоретический обзор ключевых понятий ML

### 1. EDA (Exploratory Data Analysis)
Разведочный анализ данных (EDA) — это первичный этап исследования датасета, направленный на понимание его структуры, природы и внутренних закономерностей. Он необходим для того, чтобы увидеть «полную картину» данных до того, как подавать их в алгоритмы машинного обучения. В процессе EDA исследователь проверяет типы данных, находит пропуски, выявляет аномалии и выбросы, а также изучает распределения признаков. Кроме того, на этом этапе анализируются корреляции между фичами и целевой переменной, что помогает сформировать гипотезы для дальнейшей предобработки. Без качественного EDA высок риск обучить модель на нерелевантной или искаженной информации.

---

### 2. Пропуски (NaN)
Пропущенные значения возникают по разным причинам: от сбоев при сборе и записи данных до нежелания пользователей заполнять необязательные поля анкеты. Обработка пропусков критически важна, так как большинство классических ML-моделей не умеют работать с `NaN` напрямую. Выбор стратегии зависит от природы данных: строки или столбцы с большим количеством пропусков можно просто удалить, если это не приведет к потере важной информации. В остальных случаях применяют импутацию — заполнение пропусков медианой или средним значением для числовых признаков, либо модой для категориальных. Для сложных взаимосвязей также используют заполнение с помощью моделей (например, KNN) или выделение факта отсутствия значения в отдельный бинарный признак-индикатор.

---

### 3. Кодирование категориальных признаков
Категориальные переменные содержат текстовые метки или группы, которые необходимо перевести в числовой формат для восприятия алгоритмами. При **Label Encoding** каждой уникальной категории присваивается целое число (0, 1, 2...), однако этот метод может создать ложное представление об иерархии или порядке категорий там, где его нет. В свою очередь, **One-Hot Encoding** преобразует одну категориальную колонку в $N$ бинарных столбцов (0 или 1), где $N$ — количество уникальных значений. One-Hot идеален для нелинейных категорий, но может сильно раздуть размерность датасета при высоком кардинале признака.

---

### 4. Масштабирование признаков
Масштабирование необходимо для того, чтобы признаки с большими значениями (например, доход или цена) не перевешивали признаки с малыми значениями (например, возраст) в моделях, чувствительных к расстояниям и градиентным спускам. **StandardScaler** примет распределение признака к нулевому среднему и единичному стандартному отклонению ($\mu = 0, \sigma = 1$), что отлично подходит для данных с нормальным распределением. **MinMaxScaler** сжимает значения в строго заданный диапазон (обычно от 0 до 1), сохраняя исходное распределение. Выбор зависит от алгоритма: линейным моделям и KNN масштабирование жизненно необходимо, тогда как решающие деревья устойчивы к разнице в масштабах.

---

### 5. Feature Engineering
Feature Engineering — это процесс создания новых признаков на основе имеющихся данных для повышения прогностической силы модели. Нативные данные не всегда напрямую отражают скрытые физические или логические связи, поэтому инженерная разработка признаков помогает подсветить их для алгоритма. Например, из даты рождения можно выделить чистый возраст, из адреса — объединить улицы в районы, а из размера семьи — создать бинарный флаг «путешествует один». Качественный Feature Engineering часто дает больший прирост к качеству модели, чем подбор гиперпараметров самого алгоритма.

---

### 6. Метрики классификации
Метрики необходимы для объективной оценки качества работы классификатора с разных точек зрения. **Accuracy** показывает общую долю правильных ответов, но может быть обманчивой при сильном дисбалансе классов. **Precision** (точность) измеряет, какая доля объектов, названных моделью положительными, реально таковыми является, а **Recall** (полнота) показывает, какую долю реальных положительных объектов модель смогла найти. Для балансировки между точностью и полнотой используется **F1-score**, представляющая собой их гармоническое среднее. Метрика **ROC-AUC** оценивает способность модели ранжировать объекты по вероятности принадлежности к классу, рассчитываясь как площадь под кривой соотношения True Positive Rate и False Positive Rate при различных порогах классификации.

**Основные формулы метрик:**

$$Accuracy = \frac{TP + TN}{TP + TN + FP + FN}$$

$$Precision = \frac{TP}{TP + FP} \quad \quad Recall = \frac{TP}{TP + FN}$$

$$F1 = 2 \cdot \frac{Precision \cdot Recall}{Precision + Recall}$$

---

### Сравнительный обзор методов и подходов

| Понятие / Инструмент | Основное назначение | Преимущества | Ограничения / Риски |
| :--- | :--- | :--- | :--- |
| **EDA** | Исследование структуры и закономерностей датасета | Выявляет ошибки, выбросы и скрытые паттерны на раннем этапе | Требует времени и глубокого погружения в предметную область |
| **Label Encoding** | Перевод категорий в целые числа (0, 1, 2...) | Компактность, не увеличивает количество столбцов | Может навязать модели ложный порядок/приоритет категорий |
| **One-Hot Encoding** | Создание $N$ бинарных колонок для $N$ категорий | Не навязывает порядок категорий | Проблема «проклятия размерности» при многих уникальных значениях |
| **StandardScaler** | Приведение данных к $\mu = 0, \sigma = 1$ | Эффективен для линейных моделей и алгоритмов с градиентным спуском | Чувствителен к сильным выбросам |
| **MinMaxScaler** | Сжатие данных в строго заданный интервал $[0, 1]$ | Сохраняет исходную форму распределения и нулевые значения | Стягивает основные данные в узкий диапазон при наличии выбросов |